# Segurança aplicada — Demonstração EXP-08 (offline)

**Objetivo em 2–3 minutos:** demonstrar que alerta não é prova, separar evidência de pontuação, encaminhar para revisão humana e verificar integridade de logs. **Os dados são dois casos de demonstração**. O script não consulta a blockchain, não recebe dados em tempo real, não se conecta a SIEM e nunca bloqueia transações.

Referência científica: [protocolo EXP-08](https://github.com/BorgesGusta/ethereum-mixer-detection/blob/docs/consolidacao-oficial-2026-11-17/docs/experiments/EXP-08-plano-seguranca-e-conjuntos.md).

In [ ]:
%%writefile triage.py
#!/usr/bin/env python3
"""Prototype: auditable human-in-the-loop triage of Ethereum model alerts.
No blockchain queries, no training, no automated blocking. Standard library only.
"""
import argparse
import datetime as dt
import hashlib
import json
import pathlib
import re

ADDR = re.compile(r'^0x[0-9a-fA-F]{40}$')
TX = re.compile(r'^0x[0-9a-fA-F]{64}$')
STATUSES = {'candidato', 'indeterminado', 'positivo_confirmado', 'negativo_confirmado', 'fora_do_escopo'}


def canonical(x):
    return json.dumps(x, ensure_ascii=False, sort_keys=True, separators=(',', ':'))


def validate_case(item):
    if not isinstance(item, dict): raise ValueError('Case must be object')
    contract = item.get('contract', '')
    if not ADDR.fullmatch(contract): raise ValueError('Invalid contract address')
    score = item.get('model_score')
    if score is not None and (isinstance(score, bool) or not isinstance(score, (int, float)) or not 0 <= score <= 1):
        raise ValueError('model_score must be in [0,1] or null')
    evs = item.get('evidence', [])
    if not isinstance(evs, list): raise ValueError('evidence must be list')
    for ev in evs:
        if not isinstance(ev, dict): raise ValueError('evidence item must be object')
        if ev.get('type') not in ('tx_receipt', 'contract_source', 'protocol_docs', 'review_note'):
            raise ValueError('unsupported evidence type')
        if not isinstance(ev.get('source'), str) or not ev['source'].startswith(('https://', 'http://')):
            raise ValueError('evidence source URL required')
        if 'tx_hash' in ev and not TX.fullmatch(ev['tx_hash']): raise ValueError('invalid tx hash')
    return True


def triage(item):
    """Severity = evidence-completeness workflow, not illicitness or mixer probability."""
    validate_case(item)
    evs = item.get('evidence', [])
    kinds = {e['type'] for e in evs}
    supported_tx = any(e['type']=='tx_receipt' and e.get('receipt_ok') is True and
                       e.get('pool_log_verified') is True and TX.fullmatch(e.get('tx_hash', '')) for e in evs)
    result = {
        'contract': item['contract'].lower(), 'network': item.get('network', 'ethereum-mainnet'),
        'model_score': item.get('model_score'), 'evidence_count': len(evs),
        'receipt_and_log_supported': supported_tx,
        'status': 'indeterminado',
        'priority': 'revisar_evidencias' if supported_tx else 'coletar_evidencias',
        'reason_codes': (['TX_RECEIPT_POOL_LOG'] if supported_tx else ['NO_VERIFIED_POOL_RECEIPT']) +
                        (['SOURCE_DOCUMENTED'] if 'protocol_docs' in kinds or 'contract_source' in kinds else ['SOURCE_NOT_VERIFIED']),
        'action': 'REVIEW_ONLY', 'automated_blocking': False,
        'review_required': True,
        'caution': 'An observed withdrawal, a model score or a privacy tool alone does not establish illicit activity or mixer-class equivalence.'
    }
    return result


def append_audit(out_file, source, decisions):
    """Locally hash-linked JSONL; tamper-evident when anchor hash is independently kept."""
    out_file = pathlib.Path(out_file)
    if out_file.exists() and out_file.stat().st_size: raise FileExistsError('Refuse to overwrite existing audit log')
    prev = '0'*64
    with out_file.open('x', encoding='utf-8') as stream:
        for idx, dec in enumerate(decisions, 1):
            event = {'seq': idx, 'previous_hash': prev, 'source': source, 'decision': dec}
            h = hashlib.sha256(canonical(event).encode('utf-8')).hexdigest()
            event['event_hash'] = h
            stream.write(canonical(event) + '\n')
            prev = h
    return prev


def verify_audit(path):
    prev = '0'*64
    count = 0
    for line in pathlib.Path(path).read_text(encoding='utf-8').splitlines():
        entry = json.loads(line)
        expected = entry.pop('event_hash')
        if entry['previous_hash'] != prev or hashlib.sha256(canonical(entry).encode()).hexdigest() != expected:
            raise ValueError(f'Integrity failed at line {count+1}')
        prev = expected
        count += 1
    return {'events':count, 'last_hash':prev, 'valid':True}


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('command', choices=('triage','verify'))
    p.add_argument('--input', required=True)
    p.add_argument('--output')
    args = p.parse_args()
    if args.command == 'verify':
        print(json.dumps(verify_audit(args.input), indent=2));return
    if not args.output: p.error('--output required for triage')
    raw = json.loads(pathlib.Path(args.input).read_text(encoding='utf-8'))
    if not isinstance(raw, list): raise ValueError('Input must be JSON array')
    results = [triage(e) for e in raw]
    last = append_audit(args.output, str(pathlib.Path(args.input).name), results)
    print(json.dumps({'cases':len(results), 'audit_log':args.output, 'last_hash':last, 'statuses':{'indeterminado':len(results)}}, ensure_ascii=False))

if __name__ == '__main__': main()

In [ ]:
%%writefile test_triage.py
import json
import pathlib
import tempfile
import unittest
from triage import triage, append_audit, verify_audit

ROOT = pathlib.Path(__file__).parent

class Tests(unittest.TestCase):
    def setUp(self): self.data=json.loads((ROOT/'sample_cases.json').read_text())
    def test_full_evidence_does_not_auto_label(self):
        d=triage(self.data[0]);self.assertEqual(d['status'],'indeterminado');self.assertEqual(d['priority'],'revisar_evidencias');self.assertFalse(d['automated_blocking'])
    def test_high_score_without_evidence_does_not_auto_block(self):
        d=triage(self.data[1]);self.assertEqual(d['priority'],'coletar_evidencias');self.assertFalse(d['automated_blocking'])
    def test_invalid_address(self):
        d=dict(self.data[0]);d['contract']='bad';self.assertRaises(ValueError,triage,d)
    def test_integrity_tamper(self):
        with tempfile.TemporaryDirectory() as td:
            out=pathlib.Path(td)/'audit.jsonl'
            append_audit(out,'fixture.json',[triage(e) for e in self.data]);self.assertTrue(verify_audit(out)['valid'])
            raw=out.read_text();out.write_text(raw.replace('REVIEW_ONLY','AUTO_BLOCK',1));self.assertRaises(ValueError,verify_audit,out)

if __name__=='__main__': unittest.main()

In [ ]:
%%writefile sample_cases.json
[
  {
    "contract": "0xF241d57C6DebAe225c0F2e6eA1529373C9A9C9fB",
    "network": "ethereum-mainnet",
    "model_score": null,
    "evidence": [
      {"type":"protocol_docs", "source":"https://github.com/0xbow-io/privacy-pools-core"},
      {"type":"tx_receipt", "source":"https://eth.blockscout.com/tx/0x51fdc6cf55af6598544db3cef441e405df9a24d17d7ca9e47a287828839aa786", "tx_hash":"0x51fdc6cf55af6598544db3cef441e405df9a24d17d7ca9e47a287828839aa786", "receipt_ok":true, "pool_log_verified":true}
    ]
  },
  {"contract":"0x0000000000000000000000000000000000000001", "model_score":0.82, "evidence":[]}
]

## Etapa 1 — Testes unitários e análise dos dois casos

O primeiro caso inclui evidências informadas pelo operador para um pool Privacy Pools; o segundo usa endereço de exemplo e escore hipotético de 0,82 sem evidências. Ambos devem continuar indeterminados.

In [ ]:
!python -m unittest -v test_triage
import json
from triage import triage
cases=json.load(open('sample_cases.json'))
for item in cases:
    r=triage(item)
    print('Contrato:',item['contract'],'score:',item.get('model_score'))
    print('Estado:',r['status'],'prioridade:',r['priority'],'bloqueio:',r['automated_blocking'])

## Etapa 2 — Encadear auditoria e verificar integridade

SHA-256 encadeado detecta alteração relativa ao hash terminal original **guardado externamente**. Sem ancoragem externa, um invasor pode reconstruir a cadeia inteira; isso não é um livro-razão imutável.

In [ ]:
from datetime import datetime,timezone
from pathlib import Path
from triage import append_audit,verify_audit
from shutil import copyfile
out=Path('audit_demo_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'.jsonl')
last_hash=append_audit(out,'sample_cases.json',[triage(e) for e in cases])
print('Log:',out,'SHA terminal:',last_hash)
print('Verificação:',verify_audit(out))

## Etapa 3 — Ataque demonstrativo: alteração em uma cópia do log

O procedimento abaixo **não altera o log original**, apenas uma cópia demonstrativa. A tentativa de trocar `REVIEW_ONLY` por `AUTO_BLOCK` deve gerar erro de integridade.

In [ ]:
copy=Path('copia_adulterada.jsonl')
copy.write_text(out.read_text().replace('REVIEW_ONLY','AUTO_BLOCK',1),encoding='utf-8')
try:
    verify_audit(copy)
    raise RuntimeError('Falha: adulteração não detectada')
except ValueError as error:
    print('Adulteração detectada corretamente:',error)

## Etapa 4 — Ilustração conceitual da resposta de segurança

Este diagrama **não é um grafo de transações reais**. É apenas o fluxo de decisões implementado neste protótipo. Gráficos empíricos ficam no notebook 01.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch,FancyArrowPatch
nodes=[('Entrada: contrato + evidências',.8),('Checagem de formato/indícios',.6),('Revisão humana obrigatória',.4),('JSONL + SHA-256 local',.2)]
fig,ax=plt.subplots(figsize=(9,5));ax.set_xlim(0,1);ax.set_ylim(0,1);ax.axis('off')
for text,y in nodes:
    b=FancyBboxPatch((.19,y-.055),.62,.11,boxstyle='round,pad=.016,rounding_size=.019',edgecolor='#176B78',facecolor='#E8F4F5',linewidth=2)
    ax.add_patch(b);ax.text(.5,y,text,ha='center',va='center',fontsize=12,fontweight='medium')
for j in range(len(nodes)-1):
    ax.add_patch(FancyArrowPatch((.5,nodes[j][1]-.065),(.5,nodes[j+1][1]+.065),arrowstyle='-|>',mutation_scale=16,color='#293344',lw=1.6))
ax.text(.5,.04,'SEM bloqueio automático · SEM consulta on-chain · revisão humana',ha='center',color='#AA5935',fontsize=10)
fig.savefig('fluxo_defensivo_EXP08.png',dpi=250,bbox_inches='tight');plt.show()

## Perguntas do professor

**Este protótipo detecta mixers em produção?** Não. Ele é a etapa de triagem posterior ao alerta, com dados fornecidos pelo operador.

**Um score 0,82 prova ilícito?** Não: nem a identidade funcional, nem a finalidade do contrato, nem o rótulo de crime são estabelecidos por esse número.

**O SHA-256 torna o registro imutável?** Não; requer guardar hash terminal fora do log e implantar controles de confiança/armazenamento para produção.